GUI Preprocessing: Lens Light Centre
====================================

This tool allows one to input the lens light centre(s) of a strong lens(es) via a GUI, which can be used as a fixed
value in pipelines.

__Contents__

- **Dataset:** Load and plot the strong lens dataset.
- **Search Box:** When you click on a pixel to mark a position, the search box looks around this click and finds the.
- **Clicker:** Set up the `Clicker` object from the `clicker.py` module, which monitors your mouse clicks in order.
- **Output:** Now lets plot the image and lens light centres, so we can check that the centre overlaps the.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt
from matplotlib import pyplot as plt

__Dataset__

Setup the path the datasets we'll use to illustrate preprocessing, which is the 
folder `dataset/imaging/simple`.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/simulator.py"],
        check=True,
    )

The pixel scale of the imaging dataset.

In [ ]:
pixel_scales = 0.1

Load the image which we will use to mark the lens light centre.

In [ ]:
data = al.Array2D.from_fits(
    file_path=dataset_path / "data.fits", pixel_scales=pixel_scales
)

__Search Box__

When you click on a pixel to mark a position, the search box looks around this click and finds the pixel with
the highest flux to mark the position.

The `search_box_size` is the number of pixels around your click this search takes place.

In [ ]:
search_box_size = 5

__Clicker__

Set up the `Clicker` object from the `clicker.py` module, which monitors your mouse clicks in order to determine
the lens light centres.

In [ ]:
clicker = al.Clicker(
    image=data, pixel_scales=pixel_scales, search_box_size=search_box_size
)

Set up the clicker canvas and load the GUI which you can now click on to mark the lens light centres.

In [ ]:
n_y, n_x = data.shape_native
hw = int(n_x / 2) * pixel_scales
ext = [-hw, hw, -hw, hw]
fig = plt.figure(figsize=(14, 14))
plt.imshow(data.native, cmap="jet", extent=ext)
plt.colorbar()
cid = fig.canvas.mpl_connect("button_press_event", clicker.onclick)
plt.show()
fig.canvas.mpl_disconnect(cid)
plt.close(fig)

Use the results of the Clicker GUI to create the list of lens light centres.

In [ ]:
lens_light_centres = al.Grid2DIrregular(values=clicker.click_list)

__Output__

Now lets plot the image and lens light centres, so we can check that the centre overlaps the brightest pixel in the
lens light.

In [ ]:

aplt.plot_array(array=data, title="")

Output this image of the lens light centres to a .png file in the dataset folder for future reference.

In [ ]:
aplt.plot_array(array=data, title="")

Output the lens light centres to a .json file in the dataset folder, so we can load them in modeling scripts.

In [ ]:
al.output_to_json(
    obj=lens_light_centres,
    file_path=Path(dataset_path, "lens_light_centre.json"),
)
